# Clase 078 — SVM no lineal: kernel polinomial y RBF

Cuando los datos **no son linealmente separables**, el **kernel trick** permite a `SVC`
calcular el producto interno en un espacio expandido sin materializarlo. Trabajamos los
kernels **polinomial** y **RBF (Gaussian)** sobre `make_moons` y `make_circles`, y vemos
cómo `gamma` y `C` controlan el trade-off bias/varianza.

Requiere: `numpy`, `matplotlib`, `scikit-learn`.

## 1. Datos no lineales: moons y circles

`make_moons` genera dos medialunas entrelazadas; `make_circles`, dos anillos concéntricos.
Ninguno es separable con una recta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

RND = 42
np.random.seed(RND)
Xm, ym = make_moons(n_samples=300, noise=0.20, random_state=RND)
Xc, yc = make_circles(n_samples=300, noise=0.10, factor=0.4, random_state=RND)

def plot_boundary(ax, model, X, y, title):
    h = 0.02
    x0 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
    x1 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200)
    xx, yy = np.meshgrid(x0, x1)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=18)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", edgecolor="k", s=18)
axes[0].set_title("make_moons")
axes[1].scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", edgecolor="k", s=18)
axes[1].set_title("make_circles")
plt.tight_layout(); plt.show()

## 2. El kernel lineal falla en moons

Un `SVC(kernel="linear")` no puede separar dos medialunas: la frontera es una recta.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.25, random_state=RND)
lin = SVC(kernel="linear", C=1).fit(Xtr, ytr)
acc_lin = accuracy_score(yte, lin.predict(Xte))
print(f"accuracy SVC lineal en moons: {acc_lin:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
plot_boundary(ax, lin, Xm, ym, f"Lineal (acc={acc_lin:.2f}) - inadecuado")
plt.tight_layout(); plt.show()

## 3. Kernel polinomial

`SVC(kernel="poly", degree=3, coef0=1)` captura interacciones hasta orden 3. `coef0` pesa los
términos de orden alto vs. bajo.

In [ ]:
poly = SVC(kernel="poly", degree=3, coef0=1, C=5).fit(Xtr, ytr)
acc_poly = accuracy_score(yte, poly.predict(Xte))
print(f"accuracy SVC polinomial (degree=3): {acc_poly:.3f}")
assert acc_poly > acc_lin, "el kernel polinomial debe superar al lineal en moons"

fig, ax = plt.subplots(figsize=(6, 4))
plot_boundary(ax, poly, Xm, ym, f"Poly degree=3 (acc={acc_poly:.2f})")
plt.tight_layout(); plt.show()

## 4. Kernel RBF y el efecto de gamma

`K(x, x') = exp(-γ·||x-x'||²)`. **γ alto** → frontera muy ondulada (*overfitting*);
**γ bajo** → frontera suave (*underfitting*). Lo mostramos sobre `circles`.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, gamma in zip(axes, [0.1, 1, 10, 100]):
    m = SVC(kernel="rbf", gamma=gamma, C=1).fit(Xc, yc)
    plot_boundary(ax, m, Xc, yc, f"RBF gamma={gamma}")
plt.suptitle("gamma bajo = suave (underfit) -> gamma alto = ondulado (overfit)")
plt.tight_layout(); plt.show()

## 5. GridSearchCV 2D: gamma × C + heatmap

Buscamos el mejor par `(gamma, C)` con validación cruzada y visualizamos la grilla de scores
como *heatmap*.

In [ ]:
gammas = [0.1, 1, 10]
Cs = [0.1, 1, 10]
grid = GridSearchCV(
    SVC(kernel="rbf"),
    {"gamma": gammas, "C": Cs},
    cv=3, n_jobs=1)
grid.fit(Xm, ym)
print("mejor par:", grid.best_params_)
print(f"mejor score CV: {grid.best_score_:.3f}")

scores = grid.cv_results_["mean_test_score"].reshape(len(Cs), len(gammas))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(scores, cmap="viridis", origin="lower")
ax.set_xticks(range(len(gammas))); ax.set_xticklabels(gammas)
ax.set_yticks(range(len(Cs))); ax.set_yticklabels(Cs)
ax.set_xlabel("gamma"); ax.set_ylabel("C")
for i in range(len(Cs)):
    for j in range(len(gammas)):
        ax.text(j, i, f"{scores[i, j]:.2f}", ha="center", va="center", color="w")
fig.colorbar(im, label="accuracy CV")
ax.set_title("GridSearchCV: accuracy por (gamma, C)")
plt.tight_layout(); plt.show()

## Ejercicios

1. Mostrá que `SVC(kernel="linear")` es inadecuado en `make_moons` (accuracy + frontera).
2. Entrená `SVC(kernel="poly", degree=3, coef0=1, C=5)` y comparalo con el lineal.
3. Sobre `make_circles`, variá `gamma ∈ {0.1, 1, 10, 100}` con `C` fijo y graficá las 4
   fronteras lado a lado. Identificá *underfit* y *overfit*.
4. Corré `GridSearchCV` sobre `gamma × C` con `cv` y reportá el mejor par y el *heatmap* de
   scores.

## Conclusiones

- El **kernel trick** evita materializar el *feature map* `φ(x)`: `SVC` opera en el espacio
  expandido a través de `K(x, x')`.
- **RBF** es el default robusto (un solo hiperparámetro relevante, `gamma`); **polinomial**
  conviene con interacciones de orden bajo y fijo.
- `gamma` controla la **forma** de la frontera y `C` la **regularización**; se tunean juntos
  con `GridSearchCV`.
- `SVC` es O(n²)–O(n³): no escala a >100k filas. Para eso: `LinearSVC`, `SGDClassifier` o
  aproximación de kernel (`Nystroem`).